<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [3]:
import torch.nn.functional as F

In [23]:
df = pd.read_csv('mnist_train.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([60000, 784]), torch.Size([60000]), torch.float32, torch.int64)

In [47]:
#check kamming_normal initialisation
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100) * 0.01
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10) * 0

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [48]:
print(sum(p.numel() for p in parameters))

79510


In [49]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

In [50]:
#forward pass
batch_size = 50
for i in range(20000): # Reduced iterations for faster debugging
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb = Xtr[ix]
  Yb = Ytr[ix]

  Xb_mean = Xb.mean(0, keepdim=True)
  Xb_std = Xb.std(0, keepdim=True)

  X_mean = 0.998 * X_mean + 0.002 * Xb_mean
  X_std = 0.998 * X_std + 0.002 * Xb_std

  Xb = (Xb - Xb_mean) / (Xb_std + eps)

  hpreact = Xb @ w1 + b1

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Yb)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 15000 else 0.01

  for p in parameters:
    p.data += -lr * p.grad

  if i%100 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

2.5523054599761963
0.36821162700653076
0.23992082476615906
0.371648907661438
0.2762785255908966
0.321493923664093
0.4334843158721924
0.20371116697788239
0.21071094274520874
0.3092065751552582
0.22594812512397766
0.23400245606899261
0.26891183853149414
0.10931671410799026
0.10420513898134232
0.33929333090782166
0.15023185312747955
0.33088603615760803
0.1430922895669937
0.0600060410797596
0.10355254262685776
0.1538495123386383
0.1332957148551941
0.3025881052017212
0.05488579720258713
0.05039840191602707
0.18841953575611115
0.10159854590892792
0.15831229090690613
0.12087474018335342
0.028559554368257523
0.12733988463878632
0.05516790971159935
0.11148113012313843
0.17079076170921326
0.09360367804765701
0.15225239098072052
0.05454013869166374
0.1378013640642166
0.17468811571598053
0.031247392296791077
0.15298616886138916
0.13851429522037506
0.032674096524715424
0.08041994273662567
0.07329211384057999
0.10382148623466492
0.0321987047791481
0.08519499003887177
0.041151102632284164
0.121995970

Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [51]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10000, 784])
Test data shape (labels): torch.Size([10000])


In [52]:
X_test = (X_test - X_mean) / (X_std + eps)

In [53]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(hpreact_test)
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Y_test[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 9561 
 Unmatched: 439
